# EDA 003.07 — Feature Engineering: Other Techniques

**Beyond the Kaggle Feature Engineering Course**

This notebook covers advanced feature engineering techniques not included in the [Kaggle Feature Engineering course](https://www.kaggle.com/learn/feature-engineering):

1. **Date & Time Feature Extraction** — decompose timestamps into learnable components
2. **Polynomial Features** — systematically generate interaction and power terms
3. **Lag & Window Features (Time Series)** — historical patterns as features
4. **Text-based Features** — extract signal from free-text columns
5. **Feature Selection** — identify and remove unhelpful features
6. **Binning / Discretisation** — convert continuous into categorical features

---

## Popular Datasets to Practise Feature Engineering

| Dataset | Link | Best for practising |
|---|---|---|
| House Prices (Ames) | [Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) | Interactions, transforms, target encoding |
| Titanic | [Kaggle](https://www.kaggle.com/c/titanic) | Name parsing, deck extraction, family size |
| NYC Taxi | [Kaggle](https://www.kaggle.com/c/nyc-taxi-trip-duration) | Datetime, geospatial, lag features |
| Instacart Basket | [Kaggle](https://www.kaggle.com/c/instacart-market-basket-analysis) | Aggregations, frequency, user behaviour |
| Stack Overflow Survey | [Kaggle](https://www.kaggle.com/datasets/stackoverflow/stack-overflow-2018-developer-survey) | Text features, high-cardinality categoricals |
| M5 Forecasting | [Kaggle](https://www.kaggle.com/c/m5-forecasting-accuracy) | Lag/window features, time series engineering |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import Ridge

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## 1 — Date & Time Feature Extraction

**Reference:** [Pandas datetime docs](https://pandas.pydata.org/docs/user_guide/timeseries.html)

A raw timestamp is opaque to most ML models. Breaking it into components reveals:
- **Cyclical patterns** — hourly, daily, weekly, seasonal
- **Special events** — is it a weekend? a holiday?
- **Time since reference** — account age, days since last purchase

### Cyclical Encoding
Day-of-week, hour-of-day, and month are **cyclical** — day 0 (Monday) is close to day 6 (Sunday), not far.  
Encode with sine/cosine to preserve this:

$$\sin\!\left(\frac{2\pi \cdot x}{\text{max}}\right), \quad \cos\!\left(\frac{2\pi \cdot x}{\text{max}}\right)$$

In [ ]:
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
df = pd.DataFrame({'timestamp': dates})

# Basic decomposition
df['year']        = df['timestamp'].dt.year
df['month']       = df['timestamp'].dt.month
df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0=Monday, 6=Sunday
df['day_of_year'] = df['timestamp'].dt.dayofyear
df['week']        = df['timestamp'].dt.isocalendar().week.astype(int)
df['quarter']     = df['timestamp'].dt.quarter
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
df['is_month_end']= df['timestamp'].dt.is_month_end.astype(int)

# Cyclical encoding for month and day_of_week
df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 7)

print(df[['timestamp', 'month', 'day_of_week', 'is_weekend',
          'month_sin', 'month_cos', 'dow_sin', 'dow_cos']].head(10).to_string())

# Visualise cyclical encoding
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc1 = axes[0].scatter(df['month_sin'], df['month_cos'], c=df['month'],
                       cmap='hsv', s=30, alpha=0.7)
plt.colorbar(sc1, ax=axes[0], label='Month')
axes[0].set_title('Cyclical month encoding (sin/cos)')

sc2 = axes[1].scatter(df['dow_sin'], df['dow_cos'], c=df['day_of_week'],
                       cmap='tab10', s=30, alpha=0.7)
plt.colorbar(sc2, ax=axes[1], label='Day of Week')
axes[1].set_title('Cyclical day-of-week encoding')
plt.tight_layout()
plt.show()

---
## 2 — Polynomial Features

**Reference:** [sklearn — PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)

`PolynomialFeatures` systematically creates all combinations up to degree $d$ from your feature set:

- Degree 2 from {x1, x2}: `{1, x1, x2, x1², x1·x2, x2²}`
- Degree 3: the count grows as $\binom{n+d}{d}$ — use with care on many features!

> **Tip:** Use `interaction_only=True` to get only cross-terms (no squared terms) — useful when you want interactions but not power transforms.

In [ ]:
from sklearn.model_selection import cross_val_score

np.random.seed(5)
n = 300
x1 = np.random.uniform(0, 5, n)
x2 = np.random.uniform(0, 5, n)
# True relationship involves interaction and squared terms
y  = 2*x1 + 3*x2 + 1.5*x1*x2 - 0.5*x1**2 + np.random.normal(0, 1, n)

X_raw  = np.column_stack([x1, x2])

results = {}
for degree in [1, 2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X_raw)
    cv_r2  = cross_val_score(Ridge(), X_poly, y, cv=5, scoring='r2').mean()
    results[f'degree={degree} ({X_poly.shape[1]} features)'] = cv_r2
    print(f"Degree {degree} features: {poly.get_feature_names_out()}")

print("\nCV R² scores:")
for k, v in results.items():
    print(f"  {k:40s}: {v:.4f}")

---
## 3 — Lag & Window Features (Time Series)

**Reference:** [Kaggle — Time Series Course](https://www.kaggle.com/learn/time-series)

In time series, the past is your most predictive feature. Key constructs:

| Feature type | Description | Example |
|---|---|---|
| **Lag feature** | Value at time t-k | `sales_lag_1` = yesterday's sales |
| **Rolling mean** | Average over window | `sales_rolling_7d` = 7-day average |
| **Rolling std** | Variability over window | `sales_std_7d` = recent volatility |
| **Difference** | Change from prior period | `sales_diff_1` = day-over-day change |
| **Expanding mean** | Cumulative average | `sales_cumulative_avg` |

> **Warning:** Lag features introduce NaN rows at the start of the series. Also, **never** use future data to compute features — this is leakage.

In [ ]:
np.random.seed(99)
dates = pd.date_range('2023-01-01', periods=180, freq='D')
# Simulate weekly seasonal sales
t = np.arange(180)
sales = 100 + 20 * np.sin(2 * np.pi * t / 7) + 0.3 * t + np.random.normal(0, 8, 180)
df_ts = pd.DataFrame({'date': dates, 'sales': sales})
df_ts = df_ts.set_index('date')

# Create lag and rolling features
df_ts['lag_1']       = df_ts['sales'].shift(1)
df_ts['lag_7']       = df_ts['sales'].shift(7)
df_ts['rolling_7']   = df_ts['sales'].shift(1).rolling(7).mean()   # shift first to avoid leakage
df_ts['rolling_std7']= df_ts['sales'].shift(1).rolling(7).std()
df_ts['diff_1']      = df_ts['sales'].diff(1)
df_ts['diff_7']      = df_ts['sales'].diff(7)

print(f"NaN rows after feature creation: {df_ts.isna().any(axis=1).sum()}")
print(df_ts.head(12).round(2))

# Plot
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
df_ts[['sales', 'lag_7', 'rolling_7']].dropna().plot(ax=axes[0])
axes[0].set_title('Sales with Lag and Rolling Mean Features')

df_ts['rolling_std7'].dropna().plot(ax=axes[1], color='darkorange')
axes[1].set_title('Rolling 7-day Standard Deviation (Volatility)')

plt.tight_layout()
plt.show()

---
## 4 — Text-based Features

**References:**
- [Pandas text docs](https://pandas.pydata.org/docs/user_guide/text.html)
- [sklearn — TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

When a column contains free text, you can extract:

| Feature | Description |
|---|---|
| **Word count** | Proxy for length / detail |
| **Character count** | Alternative length measure |
| **Unique word ratio** | Lexical diversity |
| **Sentiment score** | Positive / negative tone (with `textblob` or `VADER`) |
| **Keyword flags** | `contains_price`, `mentions_refund`, etc. |
| **TF-IDF vectors** | Full text representation for ML |
| **Average word length** | Readability / formality |

In [ ]:
reviews = pd.DataFrame({'review': [
    "Absolutely fantastic product! Fast shipping, great quality. Will buy again!",
    "Terrible. Broken on arrival. Waste of money. Want a refund.",
    "ok",
    "This is a decent product for the price. Nothing special but does the job.",
    "AMAZING AMAZING AMAZING love it so much! Highly recommend to everyone!!!",
    "Not what I expected. The description was misleading. Returning this.",
]})

reviews['word_count']       = reviews['review'].str.split().str.len()
reviews['char_count']       = reviews['review'].str.len()
reviews['avg_word_length']  = (reviews['review'].str.replace(r'[^\w\s]', '', regex=True)
                                                 .str.split()
                                                 .apply(lambda words: np.mean([len(w) for w in words]) if words else 0))
reviews['exclamation_count']= reviews['review'].str.count('!')
reviews['unique_word_ratio'] = (reviews['review'].str.lower().str.split()
                                                  .apply(lambda w: len(set(w)) / len(w) if w else 0))

# Keyword flags
reviews['mentions_refund']  = reviews['review'].str.contains(r'refund|return', case=False, regex=True).astype(int)
reviews['is_very_short']    = (reviews['word_count'] <= 3).astype(int)

reviews[['review', 'word_count', 'char_count', 'avg_word_length',
         'exclamation_count', 'unique_word_ratio', 'mentions_refund', 'is_very_short']]

---
## 5 — Feature Selection

**References:**
- [sklearn — Feature Selection](https://scikit-learn.org/stable/modules/feature_selection.html)
- [Feature Engineering and Selection, Ch. 6](http://www.feat.engineering/selection.html)

Adding irrelevant features can hurt performance (noise, overfitting, slower training). Feature selection identifies which to keep.

| Method | Type | Description |
|---|---|---|
| **Univariate (SelectKBest)** | Filter | Score each feature independently vs target |
| **Recursive Feature Elimination (RFE)** | Wrapper | Fit model, remove weakest feature, repeat |
| **Feature importances** | Embedded | From tree models — automatic during training |
| **Variance threshold** | Filter | Remove near-constant features |
| **Correlation filter** | Filter | Remove highly correlated pairs |

In [ ]:
from sklearn.datasets import make_regression
from sklearn.feature_selection import VarianceThreshold

# Generate dataset: 5 real features + 10 noise features
X_reg, y_reg, coef = make_regression(n_samples=300, n_features=15,
                                      n_informative=5, coef=True, random_state=42)
feature_names = [f'feat_{i}' for i in range(15)]
df_sel = pd.DataFrame(X_reg, columns=feature_names)

# --- 1. Variance Threshold (remove near-constant) ---
vt = VarianceThreshold(threshold=0.5)
vt.fit(df_sel)
print(f"After VarianceThreshold: {vt.get_support().sum()} / {df_sel.shape[1]} features retained")

# --- 2. SelectKBest (univariate F-statistic) ---
skb = SelectKBest(score_func=f_regression, k=5)
skb.fit(df_sel, y_reg)
selected_skb = [feature_names[i] for i in skb.get_support(indices=True)]
print(f"\nSelectKBest (k=5): {selected_skb}")

# --- 3. RFE ---
rfe = RFE(Ridge(), n_features_to_select=5, step=1)
rfe.fit(df_sel, y_reg)
selected_rfe = [feature_names[i] for i in rfe.get_support(indices=True)]
print(f"RFE (k=5)        : {selected_rfe}")

# True informative features
true_informative = [feature_names[i] for i in np.argsort(np.abs(coef))[-5:]]
print(f"True informative : {sorted(true_informative)}")

---
## 6 — Binning / Discretisation

**Reference:** [sklearn — KBinsDiscretizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html)

Converting a continuous feature to bins (categories) can:
- Reduce the impact of **outliers**
- Capture **non-linear, step-function** relationships
- Create **interpretable segments** (age groups, income brackets)

| Strategy | Description |
|---|---|
| **uniform** | Equal-width bins |
| **quantile** | Equal-frequency bins (each bin has same number of rows) |
| **kmeans** | Bins based on K-Means cluster boundaries |

> **Note:** Binning discards information within each bin. Use it intentionally, not as a default.

In [ ]:
np.random.seed(3)
age = np.random.exponential(scale=20, size=500) + 15  # right-skewed ages
age = np.clip(age, 15, 80)

# Manual binning with custom labels
bins   = [0, 18, 30, 45, 60, 100]
labels = ['teen', 'young_adult', 'adult', 'middle_age', 'senior']
age_group_manual = pd.cut(age, bins=bins, labels=labels)

# KBinsDiscretizer
kbd_uniform  = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
kbd_quantile = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

age_2d = age.reshape(-1, 1)
bins_uniform  = kbd_uniform.fit_transform(age_2d).ravel()
bins_quantile = kbd_quantile.fit_transform(age_2d).ravel()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(age, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Raw Age Distribution')

pd.Series(bins_uniform).value_counts().sort_index().plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Uniform bins (equal width)\n— unequal class sizes')
axes[1].set_xlabel('Bin')

pd.Series(bins_quantile).value_counts().sort_index().plot(kind='bar', ax=axes[2], color='seagreen', edgecolor='white')
axes[2].set_title('Quantile bins (equal frequency)\n— equal class sizes')
axes[2].set_xlabel('Bin')

plt.tight_layout()
plt.show()

print("Uniform bin edges :", kbd_uniform.bin_edges_[0].round(1))
print("Quantile bin edges:", kbd_quantile.bin_edges_[0].round(1))

---
## Summary — When to Use Each Technique

| Technique | Use when |
|---|---|
| **Datetime decomposition** | You have timestamp columns — always decompose them |
| **Cyclical encoding** | Day-of-week, hour, month — where the cycle wraps around |
| **Polynomial features** | You suspect interactions or curvature; small feature set |
| **Lag / rolling features** | Time series data; past predicts future |
| **Text features** | Free-text columns — word count, flags, TF-IDF |
| **SelectKBest / RFE** | Many features; need to remove noise before training |
| **Binning** | Outlier-sensitive features; want to create segments |

---

## Further Reading

- [Feature Engineering and Selection](http://www.feat.engineering/) — Kuhn & Johnson (free online book)
- [Feature Engineering for Machine Learning](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/) — Zheng & Casari
- [sklearn — Feature Extraction](https://scikit-learn.org/stable/modules/feature_extraction.html)
- [Kaggle — Time Series Course](https://www.kaggle.com/learn/time-series) — for lag/window features
- [TextBlob Sentiment Analysis](https://textblob.readthedocs.io/) — simple NLP library
- [VADER Sentiment Analysis](https://github.com/cjhutto/vaderSentiment) — social media text sentiment